# Differential accessibility with PyDESeq2

Bulk ATAC-seq analysis using BAM files and IDR peaks from the ENCODE pipeline. The notebook builds a common peak set, counts fragments, checks sample variation with PCA, and tests for differential accessibility.

Sample table and output directory need to be updated before running. 

## Setup

In [ ]:
from pathlib import Path
import gzip
import inspect
import time

import numpy as np
import pandas as pd
import pysam
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.decomposition import PCA
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

### Input files and settings

Coordinate sorted PE BAM files and IDR narrowPeak files for each sample. Update placeholders below before running

In [ ]:
SAMPLE_FILES = pd.DataFrame([
    {"sample": "sample_1", "condition": "condition_a", "bam": "/path/to/sample_1.bam", "peaks": "/path/to/sample_1.narrowPeak.gz"},
    {"sample": "sample_2", "condition": "condition_a", "bam": "/path/to/sample_2.bam", "peaks": "/path/to/sample_2.narrowPeak.gz"},
    {"sample": "sample_3", "condition": "condition_a", "bam": "/path/to/sample_3.bam", "peaks": "/path/to/sample_3.narrowPeak.gz"},
    {"sample": "sample_4", "condition": "condition_b", "bam": "/path/to/sample_4.bam", "peaks": "/path/to/sample_4.narrowPeak.gz"},
    {"sample": "sample_5", "condition": "condition_b", "bam": "/path/to/sample_5.bam", "peaks": "/path/to/sample_5.narrowPeak.gz"},
    {"sample": "sample_6", "condition": "condition_b", "bam": "/path/to/sample_6.bam", "peaks": "/path/to/sample_6.narrowPeak.gz"},
])
SAMPLES = SAMPLE_FILES["sample"].tolist()
CONDITIONS = SAMPLE_FILES.set_index("sample")["condition"]

BATCH_SIZE = 200_000
N_TOP_PEAKS = 5000
N_CPUS = 2
FDR = 0.05
LFC = 0.5
output_dir = Path("/path/to/analysis_output")

## Count fragments in a common peak set

### Check the inputs

In [ ]:
if SAMPLE_FILES["sample"].duplicated().any():
    raise ValueError("Sample IDs must be unique")
if set(CONDITIONS.unique()) != {"condition_a", "condition_b"}:
    raise ValueError("Expected both condition_a and condition_b")

peak_files = dict(zip(SAMPLES, map(Path, SAMPLE_FILES["peaks"])))
bam_files = dict(zip(SAMPLES, map(Path, SAMPLE_FILES["bam"])))

for sample in SAMPLES:
    if not peak_files[sample].is_file() or not bam_files[sample].is_file():
        raise FileNotFoundError(f"Missing BAM or peak file for {sample}")
    with pysam.AlignmentFile(str(bam_files[sample]), "rb") as bam:
        if bam.header.to_dict().get("HD", {}).get("SO") != "coordinate":
            raise ValueError(f"{sample}: expected coordinate-sorted BAM")

print(f"Located inputs for {len(SAMPLES)} samples.", flush=True)

### Merge IDR peaks across samples

In [ ]:
regions_by_chrom = {}

for sample in SAMPLES:
    with gzip.open(peak_files[sample], "rt") as handle:
        for line in handle:
            if not line.strip() or line.startswith(("#", "track", "browser")):
                continue

            chrom, start, end, *_ = line.split()
            start, end = int(start), int(end)

            if start < 0 or end <= start:
                raise ValueError(f"Invalid peak in {sample}: {line}")

            regions_by_chrom.setdefault(chrom, []).append((start, end))

merged_regions = []

for chrom in sorted(regions_by_chrom):
    merged = []

    for start, end in sorted(regions_by_chrom[chrom]):
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])

    merged_regions.extend(
        (chrom, start, end) for start, end in merged
    )

peak_table = pd.DataFrame(
    merged_regions, columns=["chrom", "start", "end"]
)

peak_table.index = pd.Index(
    [
        f"{chrom}:{start}-{end}"
        for chrom, start, end in merged_regions
    ],
    name="Peak"
)

if peak_table.empty:
    raise ValueError("No peaks found")

# Ensure all peak coordinates are valid for every BAM.
for sample in SAMPLES:
    with pysam.AlignmentFile(str(bam_files[sample]), "rb") as bam:
        sizes = dict(zip(bam.references, bam.lengths))

    for chrom, group in peak_table.groupby("chrom"):
        if chrom not in sizes or group["end"].max() > sizes[chrom]:
            raise ValueError(
                f"{sample}: peak coordinates do not match BAM reference"
            )

# Arrays for fast overlap counting.
peak_lookup = {}

for chrom, group in peak_table.groupby("chrom", sort=False):
    peak_lookup[chrom] = (
        group["start"].to_numpy(dtype=np.int64),
        group["end"].to_numpy(dtype=np.int64),
        peak_table.index.get_indexer(group.index),
    )

print(
    f"Common peak set: {len(peak_table):,} merged IDR regions",
    flush=True
)

### Count fragments overlapping peaks

In [ ]:
def count_sample(bam_path, sample):
    counts = np.zeros(len(peak_table), dtype=np.int64)
    total_fragments = 0
    report_at = 5_000_000
    started = time.monotonic()

    def add_batch(chrom, starts, ends):
        if not starts or chrom not in peak_lookup:
            return

        peak_starts, peak_ends, indices = peak_lookup[chrom]

        left = np.searchsorted(peak_ends, starts, side="right")
        right = np.searchsorted(peak_starts, ends, side="left")

        # A fragment contributes once to every region it overlaps.
        delta = (
            np.bincount(left, minlength=len(indices) + 1)
            - np.bincount(right, minlength=len(indices) + 1)
        )

        counts[indices] += np.cumsum(delta[:-1])

    with pysam.AlignmentFile(str(bam_path), "rb") as bam:
        chromosome_names = bam.references
        current_id = None
        starts, ends = [], []

        for read in bam.fetch(until_eof=True):
            if (
                not read.is_read1
                or not read.is_proper_pair
                or read.is_unmapped
                or read.mate_is_unmapped
                or read.is_secondary
                or read.is_supplementary
                or read.is_duplicate
                or read.is_qcfail
                or read.reference_id != read.next_reference_id
                or read.template_length == 0
            ):
                continue

            reference_id = read.reference_id

            if reference_id != current_id:
                if current_id is not None:
                    add_batch(
                        chromosome_names[current_id], starts, ends
                    )
                current_id = reference_id
                starts, ends = [], []

            start = min(
                read.reference_start,
                read.next_reference_start
            )
            end = start + abs(read.template_length)

            starts.append(start)
            ends.append(end)
            total_fragments += 1

            if len(starts) >= BATCH_SIZE:
                add_batch(
                    chromosome_names[current_id], starts, ends
                )
                starts, ends = [], []

            if total_fragments >= report_at:
                elapsed = (time.monotonic() - started) / 60
                print(
                    f"  {sample}: "
                    f"{total_fragments / 1e6:.0f} million fragments; "
                    f"{elapsed:.1f} minutes",
                    flush=True
                )
                report_at += 5_000_000

        if current_id is not None:
            add_batch(chromosome_names[current_id], starts, ends)

    if total_fragments == 0:
        raise ValueError(f"{sample}: no eligible fragments")

    return counts, total_fragments


# These objects remain available even if a later plotting step fails.
count_matrix = pd.DataFrame(index=peak_table.index)
fragment_totals = {}

for sample in SAMPLES:
    print(f"\nStarting {sample}...", flush=True)

    sample_counts, total = count_sample(bam_files[sample], sample)

    count_matrix[sample] = sample_counts
    fragment_totals[sample] = total

    print(
        f"Completed {sample}: {total:,} fragments assessed",
        flush=True
    )

fragment_totals = pd.Series(fragment_totals).loc[SAMPLES]

print("\nCounting finished.")
display(fragment_totals.to_frame("Fragments assessed"))

### Filter low-count peaks

In [ ]:
cpm = count_matrix.div(fragment_totals, axis="columns") * 1e6
keep = (cpm >= 1).sum(axis=1) >= 3

filtered_counts = count_matrix.loc[keep].copy()

if len(filtered_counts) < 10:
    raise ValueError("Too few regions remain after count filtering")

print(
    f"\nRegions retained after filtering: "
    f"{len(filtered_counts):,} / {len(count_matrix):,}",
    flush=True
)

## Normalize counts for PCA

In [ ]:
metadata = pd.DataFrame(
    {"condition": CONDITIONS.loc[SAMPLES].tolist()},
    index=SAMPLES,
)

dds_kwargs = {
    "counts": filtered_counts.T.astype(np.int64),
    "metadata": metadata,
    "n_cpus": N_CPUS,
}

# Support current and older PyDESeq2 constructor conventions.
parameters = inspect.signature(DeseqDataSet).parameters

if "design" in parameters:
    dds_kwargs["design"] = "~condition"
else:
    dds_kwargs["design_factors"] = "condition"

print("\nNormalizing counts and fitting VST...", flush=True)

dds = DeseqDataSet(**dds_kwargs)
dds.vst(use_design=False)

vst_counts = pd.DataFrame(
    np.asarray(dds.layers["vst_counts"]),
    index=dds.obs_names,
    columns=dds.var_names,
).loc[SAMPLES]

if not np.isfinite(vst_counts.to_numpy()).all():
    raise ValueError("Transformation produced non-finite values")

### Run PCA

In [ ]:
peak_variance = vst_counts.var(axis=0)
variable_peaks = peak_variance[peak_variance > 0]

if len(variable_peaks) < 2:
    raise ValueError("Too few variable regions for PCA")

selected_peaks = variable_peaks.nlargest(
    min(N_TOP_PEAKS, len(variable_peaks))
).index

pca_input = vst_counts.loc[:, selected_peaks]

pca = PCA(n_components=2)
coordinates = pca.fit_transform(pca_input)
variance = 100 * pca.explained_variance_ratio_

pca_scores = pd.DataFrame(
    coordinates,
    index=SAMPLES,
    columns=["PC1", "PC2"],
)

print(f"\nPCA uses {len(selected_peaks):,} variable regions.")
display(pca_scores.round(2))

### Plot PCA

In [ ]:
colors = {
    "condition_a": "#3182BD",
    "condition_b": "#DE2D26",
}

fig, ax = plt.subplots(figsize=(9, 7))

for sample in SAMPLES:
    pc1, pc2 = pca_scores.loc[sample]

    ax.scatter(
        pc1, pc2,
        marker="o" if CONDITIONS.loc[sample] == "condition_a" else "^",
        s=190,
        color=colors[CONDITIONS.loc[sample]],
        edgecolor="black",
        linewidth=1.3,
        zorder=3,
    )

    ax.annotate(
        sample,
        (pc1, pc2),
        xytext=(9, 8),
        textcoords="offset points",
        fontsize=12,
    )

for marker, label in [("o", "condition_a"), ("^", "condition_b")]:
    ax.scatter(
        [], [],
        marker=marker,
        s=130,
        facecolor="white",
        edgecolor="black",
        label=label,
    )

ax.axhline(0, color="0.75", linestyle="--", linewidth=1)
ax.axvline(0, color="0.75", linestyle="--", linewidth=1)

ax.set_xlabel(f"PC1 ({variance[0]:.1f}%)", fontsize=14)
ax.set_ylabel(f"PC2 ({variance[1]:.1f}%)", fontsize=14)
ax.set_title(
    "ATAC-seq: fragment-count PCA",
    fontsize=16,
    fontweight="bold",
    pad=15,
)

ax.legend(title="Condition", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(0.20)

fig.tight_layout()
plt.show()

## Differential accessibility with PyDESeq2

PyDESeq2 is fitted using **raw integer fragment counts**. The contrast is `condition_b` versus `condition_a`, so positive log2 fold change indicates higher accessibility in `condition_b`.

### Prepare the count matrix

In [ ]:
required = [
    "count_matrix",
    "filtered_counts",
    "peak_table",
    "fragment_totals",
]
missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Run this after BAM counting finishes in the SAME kernel. "
        f"Missing variables: {', '.join(missing)}"
    )

if not all(sample in count_matrix.columns for sample in samples):
    raise RuntimeError("Some samples are missing from the count matrix.")

# Raw integer counts: samples × peaks
dar_counts = filtered_counts.loc[:, samples].T.copy()

values = dar_counts.to_numpy()

if (
    not np.isfinite(values).all()
    or (values < 0).any()
    or not np.equal(values, np.floor(values)).all()
):
    raise ValueError(
        "DESeq2 requires non-negative integer fragment counts."
    )

dar_counts = dar_counts.astype(np.int64)

metadata = pd.DataFrame(
    {"condition": CONDITIONS.loc[samples].tolist()},
    index=samples,
)

### Fit the DESeq2 model

In [ ]:
print(
    f"Fitting DESeq2: {dar_counts.shape[1]:,} regions, "
    f"{dar_counts.shape[0]} samples...",
    flush=True,
)

kwargs = {
    "counts": dar_counts,
    "metadata": metadata,
    "n_cpus": N_CPUS,
    "refit_cooks": True,
}

# Compatibility with current and older PyDESeq2 versions
if "design" in inspect.signature(DeseqDataSet).parameters:
    kwargs["design"] = "~condition"
else:
    kwargs["design_factors"] = "condition"

dar_dds = DeseqDataSet(**kwargs)
dar_dds.deseq2()

### Test condition differences

In [ ]:
print("\nTesting condition_b versus condition_a...", flush=True)

dar_stats = DeseqStats(
    dar_dds,
    contrast=["condition", "condition_b", "condition_a"],
    alpha=FDR,
    n_cpus=N_CPUS,
)

dar_stats.summary()

results = dar_stats.results_df.copy()
results.index.name = "Peak"

### Select differential regions

In [ ]:
results["Status"] = "Not meeting DAR criteria"
results.loc[
    results["padj"].isna(), "Status"
] = "Not assessed (padj NA)"

significant = results["padj"].lt(FDR)

results.loc[
    significant & results["log2FoldChange"].ge(LFC),
    "Status",
] = "Higher in condition_b"

results.loc[
    significant & results["log2FoldChange"].le(-LFC),
    "Status",
] = "Lower in condition_b"

# Add genomic coordinates
results = peak_table.loc[
    results.index, ["chrom", "start", "end"]
].join(results)

results = results.sort_values("padj", na_position="last")

dars = results.loc[
    results["Status"].isin(["Higher in condition_b", "Lower in condition_b"])
].copy()

### Summarize results

In [ ]:
normalized_counts = pd.DataFrame(
    np.asarray(dar_dds.layers["normed_counts"]).T,
    index=dar_dds.var_names,
    columns=dar_dds.obs_names,
).loc[:, samples]

normalized_counts.index.name = "Peak"

sample_normalization = metadata.copy()
sample_normalization.index.name = "Sample"

sample_normalization["Fragments assessed"] = pd.Series(
    fragment_totals
).reindex(samples)

sample_normalization["DESeq2 size factor"] = (
    dar_dds.obs.loc[samples, "size_factors"].to_numpy()
)

dar_summary = pd.DataFrame({
    "Metric": [
        "Common regions before count filtering",
        "Regions entered into DESeq2",
        "Regions with non-missing adjusted P value",
        "Higher accessibility in condition_b",
        "Lower accessibility in condition_b",
        "Total DARs",
    ],
    "Count": [
        len(count_matrix),
        len(results),
        int(results["padj"].notna().sum()),
        int(results["Status"].eq("Higher in condition_b").sum()),
        int(results["Status"].eq("Lower in condition_b").sum()),
        len(dars),
    ],
})

print(
    "\nContrast: condition_b / condition_a"
    "\nPositive log2 fold change = higher accessibility in condition_b"
    "\nNegative log2 fold change = lower accessibility in condition_b"
    "\nDAR criteria: adjusted P < 0.05 and |log2 fold change| >= 0.5"
    "\nFold changes are unshrunken estimates."
)

print("\nDAR summary:")
display(dar_summary)

print("\nSample normalization:")
display(sample_normalization.round(3))

print("\nTop 20 regions ranked by adjusted P value:")
display(results.head(20))

### Save result tables

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

tables = {
    "condition_b_vs_condition_a_all_results.csv": results,
    "condition_b_vs_condition_a_DARs.csv": dars,
    "normalized_counts.csv": normalized_counts,
    "raw_counts.csv": count_matrix.loc[:, samples],
    "sample_normalization.csv": sample_normalization,
}

for filename, table in tables.items():
    table.to_csv(output_dir / filename)
    print(f"Saved table: {filename}", flush=True)

dar_summary.to_csv(
    output_dir / "DAR_summary.csv",
    index=False,
)

print("Saved table: DAR_summary.csv")
print(f"\nTables saved in:\n{output_dir}")

## Volcano plot

In [ ]:
palette = {
    "Not meeting DAR criteria": "#BDBDBD",
    "Not assessed (padj NA)": "#E0E0E0",
    "Lower in condition_b": "#3182BD",
    "Higher in condition_b": "#D73027",
}

volcano = results.loc[
    np.isfinite(results["log2FoldChange"])
    & np.isfinite(results["padj"])
].copy()

volcano["minus_log10_padj"] = -np.log10(
    volcano["padj"].clip(lower=1e-300)
)

fig, ax = plt.subplots(figsize=(9, 7))

for status, color in palette.items():
    subset = volcano.loc[volcano["Status"].eq(status)]

    ax.scatter(
        subset["log2FoldChange"],
        subset["minus_log10_padj"],
        s=12,
        alpha=0.55,
        color=color,
        edgecolors="none",
        label=f"{status} (n={len(subset):,})",
    )

ax.axhline(
    -np.log10(FDR),
    color="0.4", linestyle="--", linewidth=1,
)
ax.axvline(-LFC, color="0.4", linestyle="--", linewidth=1)
ax.axvline(LFC, color="0.4", linestyle="--", linewidth=1)

ax.set_xlabel("log₂ fold change (condition_b / condition_a)", fontsize=13)
ax.set_ylabel("−log₁₀ adjusted P value", fontsize=13)
ax.set_title(
    "ATAC-seq: condition_b versus condition_a",
    fontsize=16, fontweight="bold",
)

ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=10)

fig.tight_layout()
plt.show()

## MA plot

In [ ]:
ma = results.loc[
    results["baseMean"].gt(0)
    & np.isfinite(results["baseMean"])
    & np.isfinite(results["log2FoldChange"])
].copy()

ma_palette = {
    "Not meeting DAR criteria": "#BDBDBD",
    "Not assessed (padj NA)": "#E0E0E0",
    "Lower in condition_b": "#3182BD",
    "Higher in condition_b": "#D73027",
}

fig, ax = plt.subplots(figsize=(9, 7))

for status, color in ma_palette.items():
    subset = ma.loc[ma["Status"].eq(status)]

    if subset.empty:
        continue

    ax.scatter(
        subset["baseMean"],
        subset["log2FoldChange"],
        s=12,
        alpha=0.45,
        color=color,
        edgecolors="none",
        label=status,
    )

ax.set_xscale("log")

ax.axhline(0, color="0.3", linewidth=1)
ax.axhline(-LFC, color="0.5", linestyle="--", linewidth=1)
ax.axhline(LFC, color="0.5", linestyle="--", linewidth=1)

ax.set_xlabel("Mean normalized fragment count", fontsize=13)
ax.set_ylabel("log₂ fold change (condition_b / condition_a)", fontsize=13)
ax.set_title(
    "ATAC-seq: MA plot — condition_b versus condition_a",
    fontsize=16, fontweight="bold",
)

ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=10)

fig.tight_layout()
plt.show()